In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, Lasso

df = pd.read_csv("UrbanCabFare.csv")

features = [
    "Hour_of_Day",
    "Distance_km",
    "Demand_Supply_Ratio",
    "Trip_Duration",
    "Surge_Multiplier"
]
X = df[features]
y = df["Final_Fare"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

alphas = np.logspace(-4, 4, 100)

ridge_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge())
])

lasso_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Lasso(max_iter=20000))
])

ridge_grid = GridSearchCV(
    ridge_pipe,
    {"model__alpha": alphas},
    scoring="neg_root_mean_squared_error",
    cv=cv
)

lasso_grid = GridSearchCV(
    lasso_pipe,
    {"model__alpha": alphas},
    scoring="neg_root_mean_squared_error",
    cv=cv
)

ridge_grid.fit(X_train, y_train)
lasso_grid.fit(X_train, y_train)

print(f"Best Ridge alpha : {ridge_grid.best_params_['model__alpha']:.6f}")
print(f"Ridge CV RMSE    : {-ridge_grid.best_score_:.6f}")

print(f"\nBest Lasso alpha : {lasso_grid.best_params_['model__alpha']:.6f}")
print(f"Lasso CV RMSE    : {-lasso_grid.best_score_:.6f}")

Best Ridge alpha : 0.000100
Ridge CV RMSE    : 53.493965

Best Lasso alpha : 0.911163
Lasso CV RMSE    : 53.474613


In [3]:
ridge_coef = ridge_grid.best_estimator_.named_steps["model"].coef_
lasso_coef = lasso_grid.best_estimator_.named_steps["model"].coef_

coef_table = pd.DataFrame({
    "Feature": features,
    "Ridge": ridge_coef,
    "Lasso": lasso_coef
})

coef_table["Lasso_Selected"] = coef_table["Lasso"] != 0

coef_table

,Feature,Ridge,Lasso,Lasso_Selected
0,Hour_of_Day,0.329247,0.000000,False
1,Distance_km,163.124601,162.513653,True
2,Demand_Supply_Ratio,-2.692550,-0.924415,True
3,Trip_Duration,7.088547,6.857020,True
4,Surge_Multiplier,88.869662,87.123926,True
